# Interactively rendering Isaacus Legal Graph Schema (ILGS) documents
This Isaacus Cookbook teaches you how to enrich a document with [Kanon 2 Enricher](https://docs.isaacus.com/models/introduction#enrichment) into the [Isaacus Legal Graph Schema (ILGS)](https://docs.isaacus.com/ilgs) and then render the results with an interactive ILGS document viewer.

## 1. Setup
Before we begin, you will need an Isaacus account and a valid Isaacus API key, which you can obtain by following the first step of our [quickstart guide](https://docs.isaacus.com/quickstart).

After you have your API key, set up an `ISAACUS_API_KEY` environment variable with your API key as the value. This could be done by creating a `.env` file in the same directory as this notebook with the content: `ISAACUS_API_KEY=insert_your_api_key_here`.

We will now install and import our dependencies and set up our Isaacus API client.

In [ ]:
# Install dependencies.
%pip install isaacus python-dotenv requests

In [2]:
# Load dependencies.
import os
import socket
import functools
import threading

from pathlib import Path
from http.server import ThreadingHTTPServer, SimpleHTTPRequestHandler

import requests
import isaacus.types.ilgs.v1 as ilgs_v1

from dotenv import load_dotenv
from isaacus import AsyncIsaacus
from IPython.display import IFrame, display

In [3]:
# Initialize an Isaacus API client.
load_dotenv()  # Load environment variables from `.env` files if any are present.
client = AsyncIsaacus(api_key=os.getenv("ISAACUS_API_KEY"))

## 2. Enrichment
We will now fetch a document to enrich, specifically [Apple's terms of service](https://www.apple.com/au/legal/internet-services/itunes/au/terms.html), and then enrich it with Kanon 2 Enricher, saving the enriched document locally to `doc.json`.

Kanon 2 Enricher has a maximum input length of 16,384 tokens. To get around that, we'll also set our `overflow_strategy` argument to `auto`, which means that if a document exceeds the model's context window, it will automatically be split up into chunks that fit within the model's context window and then the enriched chunks will be intelligently stitched back together in order to form a single enriched document. This ensures that we capture all citations in our documents no matter where they appear.

In [4]:
# Fetch a document to enrich.
document_text = requests.get("https://examples.isaacus.com/apple-tos.txt").text

In [5]:
# Define an asynchronous helper function to enrich a single document with Kanon 2 Enricher.
async def enrich(text: str) -> ilgs_v1.document.Document:
    """Enrich a document with Kanon 2 Enricher."""

    response = await client.enrichments.create(
        model="kanon-2-enricher",
        texts=[text],
        overflow_strategy="auto",  # NOTE Setting our `overflow_strategy` to `auto` ensures that if a document exceeds the model's context window, it will automatically be split up into chunks and the results will be intelligently stitched back together to form a single enriched document, ensuring we capture all citations in our documents no matter where they appear.
    )

    # Retrieve and return the enriched document from the response.
    return response.results[0].document

In [6]:
# Enrich the document.
enriched_document = await enrich(document_text)

In [7]:
# Save the document.
_ = Path('doc.json').write_text(enriched_document.to_json())

## 3. Rendering
We will now render our enriched document with an interactive ILGS document viewer. This viewer will allow us to explore every annotation in our document, including all extracted entities, relationships, and citations.

Because our ILGS document viewer relies on features that do not always work in Jupyter notebooks, we will render the viewer by quickly spinning up a local web server to serve it.

In [8]:
# Pick a free port to serve our viewer on.
with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
    s.bind(("127.0.0.1", 0))
    port = s.getsockname()[1]

handler = functools.partial(SimpleHTTPRequestHandler, directory=Path('.').resolve())
httpd = ThreadingHTTPServer(("127.0.0.1", port), handler)
threading.Thread(target=httpd.serve_forever, daemon=True).start()

print(f'The ILGS document viewer is being served at http://127.0.0.1:{port}/viewer.html')

The ILGS document viewer is being served at http://127.0.0.1:53251/viewer.html
